# 02 Preprocessing
Crops the occipital lobe around V1-V3, builds tissue maps and surfaces, fits the reference fODF, and prepares network inputs.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
subjects = PL.make_subject_lists(cfg)
main = [PL.session(sid) for sid in dict.fromkeys(subjects["test"] + subjects["train"] + subjects["val"] + subjects["retest"])]
extra = [PL.session(sid, "7T") for sid in subjects["t7"]] + [PL.session(sid, "retest") for sid in subjects["retest"]]
with Timer(cfg, "02_crop"):
    PL.prepare_all(cfg, main + extra)

In [ ]:
# Reference fODF (multi-shell CSD on the full 1.25 mm data) and network inputs
fit_device = device if device.type == "cuda" else "cpu"
with Timer(cfg, "02_reference_fodf"):
    PL.fodf_all(cfg, main + [session for session in extra if session["kind"] == "retest"], "REF", "msmt_csd", fit_device, "reference fODF")
with Timer(cfg, "02_network_inputs"):
    PL.net_inputs_all(cfg, subjects["train"] + subjects["val"])

In [ ]:
# Quality check on one subject: tissue fractions, seeds, fODF energy
anat = PL.load_anat(cfg, main[0]); reference_fodf = PL.fodf(cfg, main[0], "REF", "msmt_csd"); z_slice = int(anat["shape"][2] // 2)
seed_vertices = np.flatnonzero(anat["area"] > 0); near_slice = np.abs(anat["verts_vox"][seed_vertices, 2] - z_slice) < 1
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(anat["pve"][0][:, :, z_slice].T, origin="lower", cmap="gray"); axes[0].set_title("white-matter fraction")
axes[0].scatter(anat["verts_vox"][seed_vertices[near_slice], 0], anat["verts_vox"][seed_vertices[near_slice], 1], c=anat["area"][seed_vertices[near_slice]], s=6, cmap="viridis")
axes[1].imshow(anat["sdist"][:, :, z_slice].T, origin="lower", cmap="coolwarm", vmin=-5, vmax=5); axes[1].set_title("distance to white surface (mm)")
axes[2].imshow(np.linalg.norm(reference_fodf["sh"][1:, :, :, z_slice], axis=0).T, origin="lower", cmap="magma"); axes[2].set_title("reference fODF anisotropic energy")
[axis.axis("off") for axis in axes]; fig.savefig(FIG / "qc_preprocessing.png", dpi=150, bbox_inches="tight")
print("subject", anat["sid"], "| crop", anat["shape"], "| V1-V3 vertices", len(seed_vertices), "| with retinotopy:", "prf_ang" in anat)